# Hopeless Loans: Ridge Regression and Batch Experiments

**Goal.** For every client who is 120 days overdue, decide whether to sue. Suing costs 500 GEL plus 10% of the outstanding balance, so it only pays off if the client later returns more than that.

**Approach**
1. Build one row per client with the 24-month recovery rate (`rr24`) as the target.
2. Train regression models that predict `rr24`.
3. Turn the prediction into a decision: sue if `predicted_rr × outstanding − cost > margin`.
4. Score the decision by realised net profit on a validation period the model never trained on.

**Contents**
1. Setup
2. Configuration
3. Data
4. Evaluation
5. Transformers
6. Ridge pipeline
7. Batch experiments

## 1. Setup

### 1.1 Install and connect Drive
Installs the tracking libraries and mounts Google Drive, where the three CSV files live.

In [1]:
%pip install -q dagshub mlflow

from google.colab import drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.0/274.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.5/231.5 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4

### 1.2 Imports

In [2]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import lightgbm as lgb
from joblib import Parallel, delayed
from scipy.stats import spearmanr


from xgboost import XGBRegressor
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

### 1.3 MLflow and DagsHub
Connects MLflow to the DagsHub repo, so every run is stored there.

Autologging is switched off because all logging is done by hand below. With autolog on, MLflow also records the pipeline's own parameters, which clashed with ours and crashed a run.

In [3]:
import dagshub

dagshub.init(repo_owner='izere23', repo_name='hopeless_loans', mlflow=True)

mlflow.autolog(disable=True)
mlflow.sklearn.autolog(disable=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=72af5087-0ed8-4ee5-9823-ff10b0a6db10&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=5b93325aed5aed0c2822c53cb168c991aeb1ac05b3336a8e5aa9c205bb02f34b




Accessing as izere23

Initialized MLflow to track repo "izere23/hopeless_loans"

Repository izere23/hopeless_loans initialized!

## 2. Configuration

| Setting | Meaning |
|---|---|
| `DATA_DIR` | Folder on Drive with the three CSV files |
| `M` | The 24 monthly payment columns |
| `ID_COLS` | Columns that identify one client in one period |
| `LEAKAGE` | Features that must never be used. `feature_045` is measured after enforcement starts |
| `FIXED_FEE`, `VAR_FEE` | Court cost: 500 GEL plus 10% of the outstanding balance |
| `FEE_MODE` | `'client'` charges the 500 once per client, `'loan'` once per loan |
| `EXCLUDE_MIXED` | `True` drops clients who also hold a secured loan |
| `VALID_START`, `TEST_START` | Dates where the validation and test periods begin |

In [4]:
DATA_DIR = Path('/content/drive/My Drive/BOG/')

M = [f'M{i:02d}_PAID' for i in range(1, 25)]
ID_COLS = ['CLIENT_KEY', 'PERIOD']
LEAKAGE = ['feature_045']


FIXED_FEE     = 500
VAR_FEE       = 0.10
FEE_MODE      = 'client'
EXCLUDE_MIXED = False
MIN_MONTHS    = 12
VALID_START   = '2024/09'
TEST_START    = '2025/03'
SPLIT_VERSION = 'v2_12m'

## 3. Data

### 3.1 Load
`loan_month_recovery.csv` has one row per loan. `Features.csv` has one row per client and period.

In [5]:
rec  = pd.read_csv(f'{DATA_DIR}/loan_month_recovery.csv')
feat = pd.read_csv(f'{DATA_DIR}/Features.csv').rename(columns={'client_key': 'CLIENT_KEY'})
print(rec.shape, feat.shape)

(70173, 33) (29763, 52)


### 3.2 Client table
The sue decision is made per client, so loans are summed up to one row per client and period.

- Only unsecured loans with a positive balance are kept
- Only cohorts with all 24 months observed are kept, so a half-observed client does not look like a bad payer
- `outst` is the total balance and `n_loans` the number of loans
- `cost` is the court cost, `rr24` is recovery divided by balance (the model's target), and `net24` is the realised profit if we had sued

In [6]:
def build_client_table(rec, feat):
    r = rec.copy()
    r['p12'] = r[M[:12]].sum(axis=1)
    r['p24'] = r[M].sum(axis=1)

    has_sec = r.groupby(ID_COLS).HAS_COLLATERAL.max().rename('has_secured').reset_index()
    L = r[(r.HAS_COLLATERAL == 0) & (r.OUTST_AT_TRIAL > 0)]

    cl = (L.groupby(ID_COLS)
            .agg(outst=('OUTST_AT_TRIAL', 'sum'),
                 n_loans=('LOAN_KEY', 'size'),
                 p12=('p12', 'sum'),
                 p24=('p24', 'sum'),
                 months_obs=('MONTHS_OBSERVED', 'min'))
            .reset_index())
    cl = cl.merge(has_sec, on=ID_COLS)

    if EXCLUDE_MIXED:
        cl = cl[cl.has_secured == 0]
    cl = cl[cl.months_obs >= MIN_MONTHS].copy()

    full = cl.months_obs == 24
    ratio = cl.loc[full, 'p24'].sum() / cl.loc[full, 'p12'].sum()
    cl['estimated'] = (~full).astype(int)
    cl['p24'] = np.where(full, cl.p24, np.minimum(ratio * cl.p12, cl.outst))

    fee_units = cl.n_loans if FEE_MODE == 'loan' else 1
    cl['cost']  = FIXED_FEE * fee_units + VAR_FEE * cl.outst
    cl['rr24']  = (cl.p24 / cl.outst).clip(0, 1)
    cl['net24'] = cl.p24 - cl.cost

    print(f"24m / 12m recovery ratio (from fully observed clients): {ratio:.3f}")
    return cl.merge(feat, on=ID_COLS, how='left')

df = build_client_table(rec, feat)
print(df.shape)
df.head()

24m / 12m recovery ratio (from fully observed clients): 1.405
(21424, 62)


,CLIENT_KEY,PERIOD,outst,n_loans,p12,p24,months_obs,has_secured,estimated,cost,...,feature_041,feature_042,feature_043,feature_044,feature_045,feature_046,feature_047,feature_048,feature_049,feature_050
0,700000,2023/04,15673.95,3,348.48,514.550000,24,0,0,2067.395,...,0.2802,0,7,1,2,1,0.6764,0,0,1
1,700001,2025/05,4064.57,3,1578.69,2217.719999,15,0,1,906.457,...,0.3527,0,1,3,5,0,0.4137,2,1,1
2,700004,2023/04,1530.03,1,0.00,0.000000,24,0,0,653.003,...,0.6906,0,5,4,2,1,0.4862,1,0,1
3,700006,2024/03,1053.29,1,0.00,0.000000,24,0,0,605.329,...,0.2851,0,3,2,0,0,0.2094,0,1,1
4,700008,2025/05,2142.92,1,0.00,0.000000,15,0,1,714.292,...,0.3162,1,3,3,0,0,0.2502,1,1,1


### 3.3 Train, validation and test split
The split follows time, not chance: train is everything before `VALID_START`, validation runs up to `TEST_START`, and test is everything after. A client who shows up in a later split is removed from the earlier ones, so the same person is never in two splits.

Test stays untouched until the final model is chosen.

`reference_points` prints three yardsticks for each split:
- `sue_all`: net profit if we sued every client
- `oracle`: net profit if we sued exactly the profitable clients, the best possible result
- `profitable_share`: the fraction of clients who are profitable to sue

In [7]:
def time_group_split(df):
    tr = df[df.PERIOD < VALID_START]
    va = df[(df.PERIOD >= VALID_START) & (df.PERIOD < TEST_START)]
    te = df[df.PERIOD >= TEST_START]
    tr = tr[~tr.CLIENT_KEY.isin(set(va.CLIENT_KEY) | set(te.CLIENT_KEY))]
    va = va[~va.CLIENT_KEY.isin(set(te.CLIENT_KEY))]
    return tr.reset_index(drop=True), va.reset_index(drop=True), te.reset_index(drop=True)

def reference_points(d):
    return dict(n=len(d), estimated_share=round(d.estimated.mean(), 3),
                sue_all=round(d.net24.sum()),
                oracle=round(d.net24.clip(lower=0).sum()),
                profitable_share=round((d.net24 > 0).mean(), 3))

train, val, test = time_group_split(df)

for name, d in [('train', train), ('valid', val), ('test', test)]:
    print(f'{name:6s}', d.PERIOD.min(), '->', d.PERIOD.max(), reference_points(d))

train  2023/01 -> 2024/08 {'n': 9434, 'estimated_share': np.float64(0.0), 'sue_all': -3208828, 'oracle': 4857736, 'profitable_share': np.float64(0.185)}
valid  2024/09 -> 2025/02 {'n': 4288, 'estimated_share': np.float64(1.0), 'sue_all': -1552098, 'oracle': 2204570, 'profitable_share': np.float64(0.183)}
test   2025/03 -> 2025/08 {'n': 5146, 'estimated_share': np.float64(1.0), 'sue_all': -2017227, 'oracle': 2519377, 'profitable_share': np.float64(0.178)}


### 3.4 Features and target
The inputs are `outst`, `n_loans`, `has_secured` and every `feature_*` column except the leakage ones. The target is `rr24`.

In [8]:
feature_cols = ['outst', 'n_loans', 'has_secured'] + \
               [c for c in df.columns if c.startswith('feature_') and c not in LEAKAGE]

X_train, y_train = train[feature_cols], train['rr24']
X_val,   y_val   = val[feature_cols],   val['rr24']
print(X_train.shape, X_val.shape)

(9434, 52) (4288, 52)


### 3.5 Data funnel
Shows how the 70k loans shrink down to the training set, so you can see where rows are lost.

In [9]:
print("loans in recovery file:", len(rec))
print("client-periods (all):", len(rec[ID_COLS].drop_duplicates()))
un = rec[(rec.HAS_COLLATERAL == 0) & (rec.OUTST_AT_TRIAL > 0)].groupby(ID_COLS).MONTHS_OBSERVED.min()
print("with unsecured balance:", len(un))
print("fully observed (24m):", int((un == 24).sum()))
print("client table (df):", len(df))
print("train / valid / test:", len(train), len(val), len(test))
print("dropped by group-aware split:", len(df) - len(train) - len(val) - len(test))
print("columns in X_train:", X_train.shape[1])

loans in recovery file: 70173
client-periods (all): 29763
with unsecured balance: 27029
fully observed (24m): 11560
client table (df): 21424
train / valid / test: 9434 4288 5146
dropped by group-aware split: 2556
columns in X_train: 52


## 4. Evaluation
The model predicts a recovery rate, but what matters is money. These three functions turn predictions into a sue decision and score it.

- `expected_net`: predicted recovery in GEL minus the court cost, per client
- `best_margin`: tries margins from −1000 to +3000 GEL and keeps the one with the highest realised profit. A client is sued only if `expected_net` is above the margin. It is chosen on validation only
- `evaluate`: scores a decision

| Metric | Meaning |
|---|---|
| `net_profit` | Realised profit from the clients we chose to sue |
| `captured` | `net_profit` divided by the oracle profit |
| `share_sued`, `n_sued` | How many clients the rule sues |
| `hit_rate` | Share of sued clients who were actually profitable |
| `auc_profitable` | How well `expected_net` ranks profitable above unprofitable clients |
| `spearman_net` | Rank correlation between `expected_net` and real profit |
| `rmse_rr` | Error of the predicted recovery rate (for reference, not the goal) |

In [10]:
def expected_net(d, pred_rr):
    return np.clip(pred_rr, 0, 1) * d.outst.values - d.cost.values

def best_margin(d, pred_rr, grid=np.arange(-1000, 3001, 50)):
    e, y = expected_net(d, pred_rr), d.net24.values
    profits = [y[e > m].sum() for m in grid]
    i = int(np.argmax(profits))
    return float(grid[i]), float(profits[i])

def evaluate(d, pred_rr, margin):
    e, y = expected_net(d, pred_rr), d.net24.values
    sue = e > margin
    oracle = np.clip(y, 0, None).sum()
    realised = y[sue].sum()
    return dict(
        net_profit=realised,
        n_sued=int(sue.sum()),
        share_sued=sue.mean(),
        hit_rate=(y[sue] > 0).mean() if sue.any() else np.nan,
        captured=realised / oracle,
        auc_profitable=roc_auc_score(y > 0, e),
        spearman_net=spearmanr(e, y)[0],
        rmse_rr=float(np.sqrt(np.mean((np.clip(pred_rr, 0, 1) - d.rr24.values) ** 2))),
    )

## 5. Transformers
Every transformer learns what it needs from the training data in `fit` and only applies it in `transform`. That keeps validation data from leaking into the preprocessing.

### 5.1 Cleaning
**MissingFlagAdder** adds a 0/1 column such as `feature_011_missing` for every column that is at least 5% empty in train. Missingness carries signal here: clients with no savings product (`feature_011` empty) or no declared income (`feature_039` empty) recover less.

In [11]:
class MissingFlagAdder(BaseEstimator, TransformerMixin):
    def __init__(self, min_missing=0.05):
        self.min_missing = min_missing

    def fit(self, X, y=None):
        miss = X.isna().mean()
        self.flag_cols_ = miss[miss >= self.min_missing].index.tolist()
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.flag_cols_:
            X[f"{col}_missing"] = X[col].isna().astype("int8")
        return X

**SignedLogTransformer** compresses heavy right tails with `sign(x) × log1p(|x|)`. It only touches columns whose skew in train is above the threshold, and skips binary columns and the missing flags. The sign is kept because some money features can be negative.

In [12]:
class SignedLogTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, skew_threshold=2.0):
        self.skew_threshold = skew_threshold

    def fit(self, X, y=None):
        cols = []
        for c in X.select_dtypes(include="number").columns:
            if c.endswith("_missing") or X[c].nunique() <= 2:
                continue
            if abs(X[c].skew()) > self.skew_threshold:
                cols.append(c)
        self.cols_ = cols
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.cols_:
            X[c] = np.sign(X[c]) * np.log1p(np.abs(X[c]))
        return X

**Winsorizer** clips each numeric column to its 1st and 99th percentile from train, so a few extreme clients cannot dominate.

In [13]:
class Winsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        self.bounds_ = {}
        for c in X.select_dtypes(include="number").columns:
            if c.endswith("_missing") or X[c].nunique() <= 2:
                continue
            self.bounds_[c] = (X[c].quantile(self.lower), X[c].quantile(self.upper))
        return self

    def transform(self, X):
        X = X.copy()
        for c, (lo, hi) in self.bounds_.items():
            X[c] = X[c].clip(lo, hi)
        return X

**DropColumns** removes columns that carry no extra information: `feature_050` is constant and `feature_033` is identical to `feature_017`.

In [14]:
class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, cols=("feature_033", "feature_050")):
        self.cols = cols

    def fit(self, X, y=None):
        self.cols_ = list(self.cols)
        return self

    def transform(self, X):
        return X.drop(columns=self.cols_, errors="ignore")

### 5.2 Preprocessing
**AutoPreprocessor** is for linear models. Numeric columns get median imputation and optional scaling. Text columns get the most frequent value and one-hot encoding. This dataset has no text columns, so only the numeric branch is active.

**NumericPassthrough** is for tree models. It keeps the numeric columns untouched, including empty values, which trees handle on their own.

In [15]:
class AutoPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, use_scaling=True, min_freq=None):
        self.use_scaling = use_scaling
        self.min_freq = min_freq
        self.preprocessor_ = None

    def fit(self, X, y=None):
        num_cols = X.select_dtypes(include="number").columns.tolist()
        cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

        num_steps = [("imputer", SimpleImputer(strategy="median"))]
        if self.use_scaling:
            num_steps.append(("scaler", StandardScaler()))
        num_pipeline = Pipeline(num_steps)

        ohe = (OneHotEncoder(handle_unknown="ignore", min_frequency=self.min_freq)
               if self.min_freq is not None else OneHotEncoder(handle_unknown="ignore"))
        cat_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", ohe)
        ])

        self.preprocessor_ = ColumnTransformer([
            ("num", num_pipeline, num_cols),
            ("cat", cat_pipeline, cat_cols)
        ])
        self.preprocessor_.fit(X, y)
        return self

    def transform(self, X):
        return self.preprocessor_.transform(X)

In [16]:
class NumericPassthrough(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.num_cols_ = X.select_dtypes(include="number").columns.tolist()
        dropped = [c for c in X.columns if c not in self.num_cols_]
        if dropped:
            print("non-numeric columns ignored:", dropped)
        return self

    def transform(self, X):
        return X[self.num_cols_].astype("float32")

### 5.3 Feature engineering
**FeatureEngineer** adds three size features: `log_outst`, `avg_loan` (balance per loan) and `outst_to_income` (balance divided by declared income, whose units are not verified, so treat it as a ranking signal). Not used by the experiments below yet.

In [17]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["log_outst"] = np.log1p(X["outst"])
        X["avg_loan"]  = X["outst"] / X["n_loans"]
        if "feature_039" in X.columns:
            income = X["feature_039"].where(X["feature_039"] > 0)
            X["outst_to_income"] = X["outst"] / income
        return X

In [18]:
class AdvancedFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X["log_outst"] = np.log1p(X["outst"])
        X["avg_loan"]  = X["outst"] / X["n_loans"]
        X["outst_x_n_loans"] = X["outst"] * X["n_loans"]
        if "feature_039" in X.columns:
            income = X["feature_039"].where(X["feature_039"] > 0)
            X["outst_to_income"] = X["outst"] / income
            X["income_per_loan"] = income / X["n_loans"]
        return X

### 5.4 Two-part estimator
**TwoPartRegressor** splits the problem in two, because about half of the clients recover nothing at all:

`predicted rr24 = P(any recovery) × expected rate given that they recover`

The first part is a classifier and the second a regressor trained only on clients who paid. The output is a plain predicted `rr24`, so `evaluate` works on it unchanged. Not used by the experiments below yet.

In [19]:
class TwoPartRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, clf=None, reg=None):
        self.clf = clf
        self.reg = reg

    def fit(self, X, y):
        y = np.asarray(y)
        pay = y > 0
        self.clf_ = clone(self.clf).fit(X, pay)
        self.reg_ = clone(self.reg).fit(X[pay], y[pay])
        return self

    def predict(self, X):
        return self.clf_.predict_proba(X)[:, 1] * np.clip(self.reg_.predict(X), 0, 1)

## 6. Ridge pipeline
A pipeline chains the steps so that one `fit` call runs them all in order, and `predict` pushes new data through the same fitted steps.

Order: cleaning, then preprocessing, then the model. This is the reference definition of a single Ridge model and it is not fitted here. Section 7 builds the same layout automatically for every experiment.

In [20]:
cleaning_pipeline = Pipeline([
    ("missing_flags", MissingFlagAdder(min_missing=0.05)),
    ("log_skewed",    SignedLogTransformer(skew_threshold=2.0)),
])

## 7. Batch experiments
Each experiment is a small dictionary (a spec) that names a cleaning recipe, a preprocessing recipe and a model. All specs are fitted in parallel, shown in one table, and only then logged to MLflow one by one.

### 7.1 Fitting
`fit_and_score` builds the full pipeline from a spec, fits it on train, tunes the margin on validation and scores both sets.

- `N_JOBS`: how many models are fitted at once. Set it to 1 if joblib complains
- `LOG_MODELS`: saving the model file made each upload take about 100 seconds and got the runtime killed. Models refit in seconds, so they are not saved here. Switch it on only for finalists
- `EXPERIMENTS`: which MLflow experiment each model family is logged to

In [21]:
N_JOBS     = 2
LOG_MODELS = False

EXPERIMENTS = {
    "ridge":      "Ridge_Training",
    "lasso":      "Lasso_Training",
    "elasticnet": "ElasticNet_Training",
    "lightgbm":   "LightGBM_Training",
    "xgboost":    "XGBoost_Training",
    "logistic_two_part": "LogisticTwoPart_Training",
}

# 7.1 fit_and_score: replace the Pipeline([...]) block with
def step(spec, key):
    f = spec.get(key)
    return f() if f else "passthrough"

def fit_and_score(spec):
    pipe = Pipeline([(k, step(spec, k)) for k in
                 ("feature_engineering", "cleaning", "preprocessing", "feature_selection", "model")])
    t0 = time.time()
    pipe.fit(X_train, y_train)
    train_pred, val_pred = pipe.predict(X_train), pipe.predict(X_val)
    margin, _ = best_margin(val, val_pred)
    return dict(spec=spec, pipeline=pipe, margin=margin,
                fit_seconds=time.time() - t0,
                m_train=evaluate(train, train_pred, margin),
                m_val=evaluate(val, val_pred, margin))

### 7.2 Printing and logging
`print_result` shows one run in plain text. `log_result` prints it, then sends it to MLflow: the settings as parameters and the scores as metrics, each in a single call (one call per item was what made logging slow). The `gap` is train `captured` minus validation `captured`, and a large positive gap means the model memorises the training data.

In [22]:
def print_result(r):
    s, m_tr, m_va = r["spec"], r["m_train"], r["m_val"]
    gap = m_tr["captured"] - m_va["captured"]
    print("=" * 70)
    print(f"{s['name']}  |  {s['family']}  |  cleaning: {s['cleaning_desc']}  |  fit {r['fit_seconds']:.0f}s")
    print(f"  margin: {r['margin']}")
    print(f"  Valid net profit: {m_va['net_profit']:,.0f} | captured: {m_va['captured']:.3f} | AUC: {m_va['auc_profitable']:.3f}")
    print(f"  Train captured  : {m_tr['captured']:.3f} | gap: {gap:.3f}")
    print(f"  Share sued: {m_va['share_sued']:.3f} ({m_va['n_sued']} clients) | hit rate: {m_va['hit_rate']:.3f}")

def log_result(r):
    s, m_tr, m_va = r["spec"], r["m_train"], r["m_val"]
    gap = m_tr["captured"] - m_va["captured"]

    print_result(r)

    exp_name = s.get("experiment", EXPERIMENTS[s["family"]])
    mlflow.set_experiment(exp_name)

    with mlflow.start_run(run_name=s["name"]):
        mlflow.set_tags({"stage": s["stage"], "family": s["family"]})

        params = {"model": s["family"], "cleaning": s["cleaning_desc"], "margin": r["margin"],
          "target": "rr24", "n_features": len(feature_cols),
          "fee_mode": FEE_MODE, "exclude_mixed": EXCLUDE_MIXED,
          "split_version": SPLIT_VERSION, "min_months": MIN_MONTHS}


        params.update({k: s[k] for k in ("prep_desc", "fe_desc", "fs_desc")})

        params.update({f"hp_{k}": str(v)[:250]
                       for k, v in r["pipeline"].named_steps["model"].get_params().items()})
        mlflow.log_params(params)

        metrics = {f"valid_{k}": float(v) for k, v in m_va.items() if np.isfinite(v)}
        metrics.update({f"train_{k}": float(v) for k, v in m_tr.items() if np.isfinite(v)})
        metrics["overfit_gap_captured"] = gap
        mlflow.log_metrics(metrics)

        if LOG_MODELS:
            mlflow.sklearn.log_model(r["pipeline"], name=s["family"], serialization_format="cloudpickle")

    print(f"  -> logged to {exp_name}")

### 7.3 Experiment specs
The three `clean_*` functions are cleaning recipes of increasing strength, and `prep_linear` / `prep_tree` are the two preprocessing recipes. They are functions so that every run gets fresh objects.

| Cleaning | Steps |
|---|---|
| `clean_flags` | missing flags |
| `clean_2b` | missing flags, then log transform |
| `clean_2c` | missing flags, log transform, then winsorizing |

Fields of a spec: `name`, `stage` and `family` (used as MLflow tags), `cleaning` and `cleaning_desc`, `preprocessing`, and `model`. To add an experiment, append another dictionary to the list.

In [23]:
from sklearn.feature_selection import SelectKBest, f_regression, SelectFromModel

def sel_k20():   return SelectKBest(f_regression, k=20)
def sel_lasso(): return SelectFromModel(Lasso(alpha=0.001, max_iter=5000))

def prep_raw():    return AutoPreprocessor(use_scaling=False)
def prep_linear(): return AutoPreprocessor(use_scaling=True)
def prep_tree():   return NumericPassthrough()

def clean(drop=False, flags=False, log=False, winsor=False):
    steps = []
    if drop:   steps.append(("drop", DropColumns()))
    if flags:  steps.append(("missing_flags", MissingFlagAdder(0.05)))
    if log:    steps.append(("log_skewed", SignedLogTransformer(2.0)))
    if winsor: steps.append(("winsorize", Winsorizer(0.01, 0.99)))
    return Pipeline(steps) if steps else "passthrough"

def make_spec(name, stage, family, model, prep, fe=False, sel=None, **ck):
    on = [k for k, v in ck.items() if v]
    return dict(name=name, stage=stage, family=family, model=model,
                preprocessing=prep, prep_desc=prep.__name__,
                cleaning=lambda: clean(**ck), cleaning_desc="+".join(on) or "none",
                feature_engineering=FeatureEngineer if fe else None, fe_desc="size_features" if fe else "none",
                feature_selection=sel, fs_desc=sel.__name__ if sel else "none")

ridge = lambda: Ridge(alpha=100.0, random_state=42)
lasso = lambda: Lasso(alpha=0.001, max_iter=5000, random_state=42)
enet  = lambda: ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000, random_state=42)
lgbm  = lambda: lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=15,
                                  min_child_samples=50, subsample=0.8, subsample_freq=1,
                                  colsample_bytree=0.7, reg_lambda=5.0,
                                  random_state=42, n_jobs=1, verbose=-1)
xgb   = lambda: XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.02,
                             subsample=0.7, colsample_bytree=0.6,
                             reg_alpha=1.0, reg_lambda=10.0,
                             objective="reg:squarederror", eval_metric="rmse",
                             random_state=42, n_jobs=1, tree_method="hist")
two_part_l2    = lambda: TwoPartRegressor(clf=LogisticRegression(C=0.1, max_iter=2000, solver="lbfgs"),
                                          reg=Ridge(alpha=10.0))
two_part_lasso = lambda: TwoPartRegressor(clf=LogisticRegression(C=1.0, max_iter=2000, solver="lbfgs"),
                                          reg=Lasso(alpha=0.001, max_iter=5000, random_state=42))

def ladder(prefix, family, model, tree=False, select=False):
    prep = prep_tree if tree else prep_linear
    full = dict(drop=True, flags=True, log=True, winsor=True)
    rows = [make_spec(f"{prefix}_0_raw", "0_raw", family, model, prep_tree if tree else prep_raw)]
    if not tree:
        rows.append(make_spec(f"{prefix}_1_preprocessed", "1_preprocessed", family, model, prep))
    rows += [
        make_spec(f"{prefix}_2a_drop_dupes", "2_cleaning", family, model, prep, drop=True),
        make_spec(f"{prefix}_2b_flags",      "2_cleaning", family, model, prep, drop=True, flags=True),
        make_spec(f"{prefix}_2c_log",        "2_cleaning", family, model, prep, drop=True, flags=True, log=True),
        make_spec(f"{prefix}_2d_winsorize",  "2_cleaning", family, model, prep, **full),
        make_spec(f"{prefix}_3_feature_eng", "3_feature_eng", family, model, prep, fe=True, **full),
    ]
    if select:
        rows += [
            make_spec(f"{prefix}_4a_select_k20",   "4_selection", family, model, prep, fe=True, sel=sel_k20,   **full),
            make_spec(f"{prefix}_4b_select_lasso", "4_selection", family, model, prep, fe=True, sel=sel_lasso, **full),
        ]
    return rows

specs = (ladder("Ridge",                 "ridge",             ridge, select=True)
       + ladder("Lasso",                 "lasso",             lasso)
       + ladder("ElasticNet",            "elasticnet",        enet)
       + ladder("LGBM",                  "lightgbm",          lgbm, tree=True)
       + ladder("XGB_Reg",               "xgboost",           xgb, tree=True)
       + ladder("LogisticTwoPart_L2",    "logistic_two_part", two_part_l2)
       + ladder("LogisticTwoPart_Lasso", "logistic_two_part", two_part_lasso))
print(len(specs), "experiments queued")

49 experiments queued


### 7.4 Run all experiments
Fits every spec in parallel and prints one comparison table, best validation profit first. Nothing is sent to MLflow yet, so the results are visible even if logging fails.

In [24]:
t0 = time.time()
results = Parallel(n_jobs=N_JOBS, verbose=10)(delayed(fit_and_score)(s) for s in specs)
print(f"fitted {len(results)} models in {time.time() - t0:.0f}s\n")

summary = pd.DataFrame([{
    "run": r["spec"]["name"],
    "margin": r["margin"],
    "valid_profit": round(r["m_val"]["net_profit"]),
    "valid_captured": round(r["m_val"]["captured"], 3),
    "valid_auc": round(r["m_val"]["auc_profitable"], 3),
    "hit_rate": round(r["m_val"]["hit_rate"], 3),
    "share_sued": round(r["m_val"]["share_sued"], 3),
    "n_sued": r["m_val"]["n_sued"],
    "train_captured": round(r["m_train"]["captured"], 3),
    "gap": round(r["m_train"]["captured"] - r["m_val"]["captured"], 3),
} for r in results]).sort_values("valid_profit", ascending=False)

print(summary.to_string(index=False))
print("\nvalidation clients:", len(val), "| oracle:", round(val.net24.clip(lower=0).sum()))

[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   1 tasks      | elapsed:    2.4s
[Parallel(n_jobs=2)]: Done   4 tasks      | elapsed:    2.8s
[Parallel(n_jobs=2)]: Done   9 tasks      | elapsed:    5.3s
[Parallel(n_jobs=2)]: Done  14 tasks      | elapsed:    6.7s
[Parallel(n_jobs=2)]: Done  21 tasks      | elapsed:    9.4s
[Parallel(n_jobs=2)]: Done  28 tasks      | elapsed:   17.8s
[Parallel(n_jobs=2)]: Done  37 tasks      | elapsed:   31.1s
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:   40.0s
[Parallel(n_jobs=2)]: Done  49 out of  49 | elapsed:   44.2s finished


fitted 49 models in 45s

                                 run  margin  valid_profit  valid_captured  valid_auc  hit_rate  share_sued  n_sued  train_captured    gap
        LogisticTwoPart_Lasso_2c_log    50.0        939084           0.426      0.760     0.492       0.140     602           0.402 -0.024
                   LGBM_2d_winsorize    50.0        934876           0.424      0.758     0.482       0.161     691           0.566  0.142
                         LGBM_2c_log    50.0        931481           0.423      0.755     0.478       0.161     690           0.562  0.139
                       LGBM_2b_flags    50.0        931481           0.423      0.755     0.478       0.161     690           0.562  0.139
           LogisticTwoPart_L2_2c_log   100.0        931077           0.422      0.757     0.500       0.132     568           0.403 -0.020
                  LGBM_2a_drop_dupes     0.0        930053           0.422      0.757     0.475       0.174     747           0.566  0.144
  

### 7.5 What the cleaning steps learned
For each run, lists which columns got a missing flag and how many columns were log-transformed or clipped. These are decided on train only.

In [28]:
for r in results:
    if not hasattr(cl, "named_steps"): print(f"{r['spec']['name']:20s} no cleaning"); continue
    cl = r["pipeline"].named_steps["cleaning"]
    parts = []
    if "missing_flags" in cl.named_steps:
        parts.append(f"flags={cl.named_steps['missing_flags'].flag_cols_}")
    if "log_skewed" in cl.named_steps:
        parts.append(f"log={len(cl.named_steps['log_skewed'].cols_)} cols")
    if "winsorize" in cl.named_steps:
        parts.append(f"clip={len(cl.named_steps['winsorize'].bounds_)} cols")
    print(f"{r['spec']['name']:20s}", " | ".join(parts))

Ridge_0_raw          no cleaning
Ridge_1_preprocessed no cleaning
Ridge_2a_drop_dupes  no cleaning
Ridge_2b_flags       no cleaning
Ridge_2c_log         no cleaning
Ridge_2d_winsorize   no cleaning
Ridge_3_feature_eng  no cleaning
Ridge_4a_select_k20  no cleaning
Ridge_4b_select_lasso no cleaning
Lasso_0_raw          no cleaning
Lasso_1_preprocessed no cleaning
Lasso_2a_drop_dupes  no cleaning
Lasso_2b_flags       no cleaning
Lasso_2c_log         no cleaning
Lasso_2d_winsorize   no cleaning
Lasso_3_feature_eng  no cleaning
ElasticNet_0_raw     no cleaning
ElasticNet_1_preprocessed no cleaning
ElasticNet_2a_drop_dupes no cleaning
ElasticNet_2b_flags  no cleaning
ElasticNet_2c_log    no cleaning
ElasticNet_2d_winsorize no cleaning
ElasticNet_3_feature_eng no cleaning
LGBM_0_raw           no cleaning
LGBM_2a_drop_dupes   no cleaning
LGBM_2b_flags        no cleaning
LGBM_2c_log          no cleaning
LGBM_2d_winsorize    no cleaning
LGBM_3_feature_eng   no cleaning
XGB_Reg_0_raw        no cl

### 7.6 Log to MLflow
Sends the runs to DagsHub one at a time. Logging is sequential on purpose, because parallel writes tend to clash.

In [ ]:
for r in results:
    log_result(r)
print("done:", len(results), "runs logged")

### 7.7 Leaderboard
Reads every experiment back from MLflow and ranks the finished runs by validation net profit.

In [30]:
exp_ids = [e.experiment_id for e in mlflow.search_experiments()]
runs = mlflow.search_runs(experiment_ids=exp_ids, order_by=["metrics.valid_net_profit DESC"])
runs = runs[runs["status"] == "FINISHED"]
if "params.split_version" in runs.columns:
    runs = runs[runs["params.split_version"] == SPLIT_VERSION]
cols = ["tags.mlflow.runName", "tags.stage", "tags.family", "metrics.valid_net_profit",
        "metrics.valid_captured", "metrics.valid_auc_profitable", "metrics.valid_share_sued",
        "metrics.valid_hit_rate", "metrics.overfit_gap_captured", "params.margin"]
board = runs[[c for c in cols if c in runs.columns]].copy()
board.columns = [c.split(".")[-1] for c in board.columns]
print(board.round(3).to_string(index=False))

                          runName          stage            family  valid_net_profit  valid_captured  valid_auc_profitable  valid_share_sued  valid_hit_rate  overfit_gap_captured margin
                LGBM_2d_winsorize     2_cleaning          lightgbm        934875.660           0.424                 0.758             0.161           0.482                 0.142   50.0
                      LGBM_2c_log     2_cleaning          lightgbm        931480.733           0.423                 0.755             0.161           0.478                 0.139   50.0
                    LGBM_2b_flags     2_cleaning          lightgbm        931480.733           0.423                 0.755             0.161           0.478                 0.139   50.0
        LogisticTwoPart_L2_2c_log     2_cleaning logistic_two_part        931077.222           0.422                 0.757             0.132           0.500                -0.020  100.0
               LGBM_2a_drop_dupes     2_cleaning          lightgbm    

# Test

In [31]:
finalists = ["Ridge_2c_log", "LogisticTwoPart_L2_2c_log", "LGBM_2d_winsorize"]
by_name = {r["spec"]["name"]: r for r in results}
X_test = test[feature_cols]

val_p  = {n: by_name[n]["pipeline"].predict(X_val)  for n in finalists}
test_p = {n: by_name[n]["pipeline"].predict(X_test) for n in finalists}
val_p["BLEND"]  = np.mean([val_p[n]  for n in finalists], axis=0)
test_p["BLEND"] = np.mean([test_p[n] for n in finalists], axis=0)

rows = []
for n in val_p:
    m, _ = best_margin(val, val_p[n])
    for rule, mg in [("val-tuned margin", m), ("margin 0", 0.0)]:
        e = evaluate(test, test_p[n], mg)
        rows.append(dict(model=n, rule=rule, margin=mg, test_profit=e["net_profit"],
                         captured=e["captured"], share_sued=e["share_sued"], hit_rate=e["hit_rate"]))

print(f"TEST  sue all: {test.net24.sum():,.0f} | oracle: {test.net24.clip(lower=0).sum():,.0f}")
pd.DataFrame(rows).round(3)

TEST  sue all: -2,017,227 | oracle: 2,519,377


,model,rule,margin,test_profit,captured,share_sued,hit_rate
0,Ridge_2c_log,val-tuned margin,200.0,820422.591,0.326,0.120,0.488
1,Ridge_2c_log,margin 0,0.0,858343.886,0.341,0.173,0.463
2,LogisticTwoPart_L2_2c_log,val-tuned margin,100.0,902427.139,0.358,0.133,0.510
3,LogisticTwoPart_L2_2c_log,margin 0,0.0,913821.037,0.363,0.158,0.487
4,LGBM_2d_winsorize,val-tuned margin,50.0,1021563.352,0.405,0.163,0.494
5,LGBM_2d_winsorize,margin 0,0.0,1045547.986,0.415,0.175,0.489
6,BLEND,val-tuned margin,250.0,920653.344,0.365,0.110,0.516
7,BLEND,margin 0,0.0,1004866.708,0.399,0.168,0.486


In [32]:
fd = pd.read_csv(f'{DATA_DIR}/feature_dictionary.csv').set_index('column')
best = by_name["LGBM_2d_winsorize"]["pipeline"]
pred = best.predict(X_val)
base = evaluate(val, pred, 0.0)["captured"]
rng = np.random.default_rng(0)

# 1) permutation importance: how much oracle-capture drops when a feature is shuffled
rows = []
for c in feature_cols:
    d = []
    for _ in range(3):
        Xp = X_val.copy(); Xp[c] = rng.permutation(Xp[c].values)
        d.append(base - evaluate(val, best.predict(Xp), 0.0)["captured"])
    rows.append((c, np.mean(d)))
imp = pd.DataFrame(rows, columns=["feature", "captured_drop"]).sort_values("captured_drop", ascending=False)
imp["group"] = imp.feature.map(fd.group); imp["description"] = imp.feature.map(fd.description)
display(imp.head(12).round(4))

# 2) does the predicted profit match reality? (by decile of expected net)
e = expected_net(val, pred)
t = pd.DataFrame({"decile": pd.qcut(e, 10, labels=False, duplicates="drop"), "pred": e, "actual": val.net24.values})
display(t.groupby("decile").agg(n=("actual", "size"), pred_net=("pred", "mean"), actual_net=("actual", "mean"),
                                 share_profitable=("actual", lambda s: (s > 0).mean())).round(1))

# 3) where does it work / fail? by debt size
b = pd.cut(val.outst, [0, 2000, 4000, 8000, 15000, np.inf])
display(pd.DataFrame({"b": b, "e": e, "y": val.net24.values}).groupby("b", observed=True)
        .apply(lambda g: pd.Series({"n": len(g), "sued": (g.e > 0).mean(), "profit": g.y[g.e > 0].sum()})).round(2))

,feature,captured_drop,group,description
2,has_secured,0.1841,NaN,NaN
41,feature_039,0.1091,declared income,Declared monthly income stated at the client's...
33,feature_031,0.0641,credit bureau history,Credit bureau score at the last refresh before...
21,feature_019,0.0627,repayment history on other loans,Share of scheduled instalments paid late over ...
36,feature_034,0.0369,credit bureau history,Internal behavioural score at the last monthly...
23,feature_021,0.0346,repayment history on other loans,Number of times 30+ days past due on any facil...
5,feature_003,0.0342,transaction activity,"Total credit turnover across all accounts, mon..."
38,feature_036,0.0323,credit bureau history,Share of total bureau-reported exposure that i...
13,feature_011,0.0311,account balances,Average end-of-month savings and deposit balan...
14,feature_012,0.0307,account balances,Maximum end-of-day total balance across all ac...


,n,pred_net,actual_net,share_profitable
decile,,,,
0,429,-1723.9,-1699.3,0.1
1,429,-919.8,-994.4,0.1
2,429,-709.8,-704.7,0.1
3,428,-600.3,-657.2,0.1
4,429,-529.4,-545.6,0.1
5,429,-467.6,-547.3,0.1
6,428,-368.9,-392.1,0.2
7,429,-207.6,-205.6,0.3
8,429,109.2,137.9,0.4


/tmp/ipykernel_959/1070032053.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({"n": len(g), "sued": (g.e > 0).mean(), "profit": g.y[g.e > 0].sum()})).round(2))


,n,sued,profit
b,,,
"(0.0, 2000.0]",1072.0,0.00,-94.82
"(2000.0, 4000.0]",899.0,0.08,5464.56
"(4000.0, 8000.0]",1042.0,0.23,94646.12
"(8000.0, 15000.0]",781.0,0.31,205174.73
"(15000.0, inf]",494.0,0.38,622105.65
